[JobSpy Library](https://github.com/speedyapply/JobSpy)
proxy list

In [2]:
#%pip install -U python-jobspy
#%pip install ipyleaflet # python package for interactive maps in jupyter notebooks
#%pip install xyzservices # for basemaps

import csv
from jobspy import scrape_jobs
from ipyleaflet import Map, Marker, basemaps, MarkerCluster, Popup
from ipywidgets import HTML
import pandas as pd

#import xyzservices.providers as xyz # check if this is needed

In [4]:
# CONTROL PANEL
#IO Controls
saveCSV = False
fileName = "250814_1000jobs.csv"
filePath = "C:/users/broge/Desktop/"

#Job scraper controls
sitesToSearch = "linkedin" #"indeed", "zip_recruiter", "google", "glassdoor", "bayt", "naukri", "bdjobs"
searchStr = '"engineer" NOT ("electrical engineer" OR "plumbing" OR "front end" OR "sales" OR "software" OR "full stack" OR "customer support" OR "integration" OR "civil" OR "backend" OR "devops" OR "machine learning" OR "geotechnical")'
locationToSearch = "USA"
numResults = 200 #number of postings to scrape. Roughly 52s/100jobs
postingAge = 168 #max posting age in hours

#Map controls
center = (38,-95) #center map on US
myBasemap = basemaps.OpenTopoMap #This is the type of map eg street, topographical, nighttime, etc. other options:basemaps.OpenStreetMap.Mapnik, basemaps.NASAGIBS.ViirsEarthAtNight2012

#initialize empty dataframe
jobs = pd.DataFrame()

In [10]:
jobs = scrape_jobs(
    site_name = sitesToSearch,
    search_term =searchStr,
    location= locationToSearch,
    results_wanted= numResults,
    hours_old=postingAge,
    proxies=["5.10.246.207:80","localhost"]
)
print(f"Found {len(jobs)} jobs")

if saveCSV:
    try: 
        jobs.to_csv(f"{filePath}{fileName}", mode='x', quoting=csv.QUOTE_NONNUMERIC, escapechar="\\", index=False)
        print(f"File '{filePath}{fileName}' created successfully.")
    except FileExistsError:
        print(f"File '{filePath}{fileName}' already exists. Overwriting prevented.")

2025-08-21 00:45:49,457 - ERROR - JobSpy:LinkedIn - LinkedIn: HTTPSConnectionPool(host='www.linkedin.com', port=443): Max retries exceeded with url: /jobs-guest/jobs/api/seeMoreJobPostings/search?keywords=%22engineer%22+NOT+%28%22electrical+engineer%22+OR+%22plumbing%22+OR+%22front+end%22+OR+%22sales%22+OR+%22software%22+OR+%22full+stack%22+OR+%22customer+support%22+OR+%22integration%22+OR+%22civil%22+OR+%22backend%22+OR+%22devops%22+OR+%22machine+learning%22+OR+%22geotechnical%22%29&location=USA&distance=50&pageNum=0&start=0&f_TPR=r604800 (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 400 Bad Request')))


Found 0 jobs


In [21]:
def get_coords(cityStateStr):
    """
    This function retrieves the latitude and longitude of a city.
    The job scraper returns a location for each job as a string with the format "city, SS" where SS is a two letter state abbrev.
    The state and city are looked up in a table of geographic data of US cities.
    """
    
    [city,state] = cityStateStr.split(", ")

    file_path = r"C:\Users\broge\Documents\GitHub\job-map\uscities.csv"
    with open(file_path, 'r', newline='') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            if city == row["city"] and state == row["state_id"]:
                return float(row["lat"]), float(row["lng"])
    return ""

#if a jobs dataframe doesn't exist (ie the scraper wasn't just used) assume the user already has a scraped csv
if len(jobs)==0:
    jobs = pd.read_csv(f"{filePath}{fileName}")
    jobs = jobs.fillna('') #if importing a csv, convet any NaN floats to empty strings

#To plot a marker for each job on a map, we want job title, url, company, and location (latitude+longitude)
markerList = []
for i in range(len(jobs.location)):
    #extract relevant info for each entry of jobs dataframe
    locStr = jobs.location[i]
    url = jobs.job_url[i]
    title = jobs.title[i]
    company = jobs.company[i]

    #check if the entry has a location listed; if so then look up the lat,long of the city
    if locStr:
        coords = get_coords(locStr)
        currentMarker = Marker(location=coords,draggable=False)
        currentMessage = HTML()
        currentMessage.value = f"<a href=\"{url}\">{title}</a><br>{company}"
        currentMarker.popup = currentMessage

        markerList.append(currentMarker)

#Generate map with list of markers from above created above
map = Map(basemap=myBasemap, center=center, zoom=4)
cluster = MarkerCluster(markers=markerList,max_cluster_radius=1) #default radius = 80 pixels
map.add(cluster)

Map(center=[38, -95], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…